<a href="https://colab.research.google.com/github/CamiloVga/IA-Codes/blob/main/Sistema_MultiAgente_CrewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MultiAgente de Redacción de un periódico

In [ ]:
# Instalaciones necesarias
!pip install crewai tavily-python openai langchain-openai

import os
from google.colab import userdata
from tavily import TavilyClient
from openai import OpenAI
from crewai import Agent, Task, Crew
from crewai.tools import tool
from langchain_openai import ChatOpenAI
from urllib.parse import urlparse
import re

# =============================================================================
# CONFIGURACIÓN GLOBAL
# =============================================================================

# Configurar OpenAI API Key
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Modelos OpenAI para cada agente (usando ChatGPT-4o-mini)
OPENAI_MODELS = {
    "research": "gpt-4o-mini",     # Investigación
    "writing": "gpt-4o-mini",      # Escritura
    "editing": "gpt-4o-mini",      # Edición
    "social": "gpt-4o-mini"        # Social media
}

# Configuración Tavily
TAVILY_CONFIG = {
    "max_results": 5,
    "topic": "general",
    "location": "CO",
    "language": "es",
    "include_answer": "basic"
}

# =============================================================================
# HERRAMIENTAS Y TRACKER
# =============================================================================

# Variable global para el tracker
global_tracker = None

class NewsroomTracker:
    """Gestiona fuentes y referencias"""
    def __init__(self):
        self.sources = []
        self.counter = 0

    def add_source(self, title, url, content, agent_name):
        self.counter += 1
        self.sources.append({
            'id': self.counter,
            'title': title,
            'url': url,
            'content': content[:200] + "..." if len(content) > 200 else content,
            'domain': urlparse(url).netloc if url else "Unknown",
            'found_by': agent_name
        })
        return self.counter

    def get_sources_summary(self):
        if not self.sources:
            return "Sin fuentes disponibles"

        summary = f"\n📚 FUENTES CONSULTADAS ({len(self.sources)})\n" + "="*50 + "\n"
        for s in self.sources:
            summary += f"[{s['id']}] {s['title']} | {s['domain']}\n"
            summary += f"🔗 {s['url']}\n"
            summary += f"📄 {s['content']}\n\n"
        return summary

@tool("search_web")
def search_web_tool(query: str) -> str:
    """Busca información en web usando Tavily API - LIMITADO A 2 RESULTADOS"""
    global global_tracker
    try:
        tavily = TavilyClient(api_key=userdata.get('TAVILY_KEY'))

        # Forzar explícitamente max_results = 2
        search_params = {
            "query": query,
            "max_results": 2,  # FORZADO A 2
            "topic": "general",
            "location": "CO",
            "language": "es",
            "include_answer": "basic"
        }

        print(f"🔍 Ejecutando búsqueda con max_results = {search_params['max_results']}")
        response = tavily.search(**search_params)

        results = response.get("results", [])
        print(f"✅ Búsqueda: '{query}' | Resultados obtenidos: {len(results)} | Máximo configurado: {search_params['max_results']}")

        content = f"🔍 Resultados para: {query} (máx: {search_params['max_results']})\n\n"
        for i, result in enumerate(results, 1):
            source_id = global_tracker.add_source(
                result.get('title', 'Sin título'),
                result.get('url', ''),
                result.get('content', ''),
                'Investigador'
            )
            content += f"**[{source_id}] Resultado {i}/{len(results)}: {result.get('title', '')}**\n"
            content += f"{result.get('content', '')[:400]}...\n\n"

        return content
    except Exception as e:
        return f"Error en búsqueda: {e}"

# =============================================================================
# CONFIGURACIÓN DE AGENTES
# =============================================================================

def setup_newsroom_agents():
    """Configura los 4 agentes especializados con OpenAI"""

    # Configurar LLMs con ChatOpenAI
    llms = {}
    for role, model in OPENAI_MODELS.items():
        llms[role] = ChatOpenAI(
            model=model,
            temperature=0.1 if role in ['research', 'editing'] else 0.7,
            api_key=userdata.get('OPENAI_API_KEY')
        )
        print(f"✅ Modelo OpenAI {model} configurado para {role}")

    # 1. AGENTE INVESTIGADOR
    research_agent = Agent(
        role='Investigador Senior',
        goal='Encontrar información completa y verificada sobre cualquier tema',
        backstory="""Eres un investigador periodístico experto con 15 años de experiencia.
        Sabes distinguir fuentes confiables de rumores y encontrar información precisa.
        Tu especialidad es buscar datos verificados y contexto histórico relevante.""",
        verbose=True,
        allow_delegation=False,
        llm=llms['research'],
        tools=[search_web_tool]
    )

    # 2. AGENTE REDACTOR
    writing_agent = Agent(
        role='Redactor Principal',
        goal='Crear artículos periodísticos claros, informativos y atractivos',
        backstory="""Eres un redactor senior especializado en periodismo digital.
        Escribes con estilo claro y directo, estructuras información de manera lógica
        y siempre incluyes referencias apropiadas a las fuentes consultadas.""",
        verbose=True,
        allow_delegation=False,
        llm=llms['writing']
    )

    # 3. AGENTE EDITOR
    editing_agent = Agent(
        role='Editor Jefe',
        goal='Revisar y mejorar la calidad editorial del contenido',
        backstory="""Eres el editor jefe con 20 años de experiencia en medios digitales.
        Tu ojo crítico detecta inconsistencias, errores factuales y problemas de estilo.
        Garantizas que cada publicación cumpla los estándares editoriales más altos.""",
        verbose=True,
        allow_delegation=False,
        llm=llms['editing']
    )

    # 4. AGENTE REDES SOCIALES
    social_agent = Agent(
        role='Community Manager',
        goal='Crear contenido viral y engaging para redes sociales',
        backstory="""Eres especialista en redes sociales que entiende las tendencias digitales.
        Sabes crear hooks atractivos, usar hashtags estratégicos y adaptar mensajes
        para cada plataforma. Tu contenido genera engagement y conversación.""",
        verbose=True,
        allow_delegation=False,
        llm=llms['social']
    )

    return research_agent, writing_agent, editing_agent, social_agent

# =============================================================================
# DEFINICIÓN DE TAREAS
# =============================================================================

def create_newsroom_tasks(agents, topic):
    """Crea las tareas para el flujo de trabajo"""

    research_agent, writing_agent, editing_agent, social_agent = agents

    # TAREA 1: INVESTIGACIÓN PROFUNDA
    research_task = Task(
        description=f"""
        Investiga a fondo sobre: {topic}

        Debes realizar múltiples búsquedas para cubrir:
        1. Información básica y contexto general
        2. Datos específicos y cifras actualizadas
        3. Diferentes perspectivas del tema
        4. Fuentes oficiales y expertos en el tema
        5. Contexto histórico si es relevante

        Para cada búsqueda, analiza los resultados y extrae:
        - Información clave con referencias [1], [2], etc.
        - Datos verificables y fechas importantes
        - Diferentes puntos de vista
        - Fuentes primarias cuando sea posible

        Entrega un informe estructurado con:
        - Resumen ejecutivo (2-3 párrafos)
        - Hallazgos principales organizados por temas
        - Referencias numeradas a todas las fuentes
        - Conclusiones preliminares
        """,
        agent=research_agent,
        expected_output="Informe de investigación detallado con múltiples fuentes verificadas"
    )

    # TAREA 2: REDACCIÓN DEL ARTÍCULO
    writing_task = Task(
        description=f"""
        Basándote en la investigación completa, redacta un artículo periodístico profesional sobre: {topic}

        El artículo debe incluir:

        1. **TITULAR**: Atractivo, preciso y que capte la atención
        2. **LEAD** (primer párrafo): Responde las 5W - qué, quién, cuándo, dónde, por qué
        3. **DESARROLLO**:
           - Estructura de pirámide invertida (información más importante primero)
           - Subtítulos para facilitar la lectura
           - Párrafos cortos (máximo 3 líneas cada uno)
           - Citas y referencias numeradas [1], [2], etc.
        4. **CONCLUSIÓN**: Resumen de puntos clave y perspectivas futuras

        **Estilo periodístico:**
        - Lenguaje claro, directo y objetivo
        - Voz activa
        - Información verificable y atribuida
        - Tono profesional pero accesible
        - Transiciones fluidas entre párrafos

        **Extensión**: 600-900 palabras
        **Referencias**: Incluir todas las fuentes como [1], [2], etc.
        """,
        agent=writing_agent,
        expected_output="Artículo periodístico completo y bien estructurado",
        context=[research_task]
    )

    # TAREA 3: EDICIÓN Y CORRECCIÓN
    editing_task = Task(
        description=f"""
        Revisa meticulosamente y mejora el artículo sobre: {topic}

        **VERIFICACIÓN EDITORIAL:**
        1. **Precisión factual**: Confirma datos, fechas y cifras
        2. **Gramática y ortografía**: Corrección completa
        3. **Estilo y claridad**: Mejora la fluidez del texto
        4. **Estructura**: Verifica lógica y coherencia
        5. **Referencias**: Asegura uso correcto de fuentes [1], [2], etc.
        6. **Estándares editoriales**: Cumplimiento de normas periodísticas

        **MEJORAS A IMPLEMENTAR:**
        - Fortalece el titular si es necesario
        - Mejora las transiciones entre párrafos
        - Optimiza la claridad de las ideas
        - Verifica que el lead responda todas las 5W
        - Asegura balance entre información y legibilidad

        **ENTREGA:**
        1. Artículo final editado y corregido
        2. Lista detallada de todos los cambios realizados
        3. Justificación de las mejoras implementadas
        4. Recomendaciones adicionales si las hay
        """,
        agent=editing_agent,
        expected_output="Artículo final editado + lista completa de mejoras implementadas",
        context=[writing_task]
    )

    # TAREA 4: CONTENIDO PARA REDES SOCIALES
    social_task = Task(
        description=f"""
        Crea una estrategia completa de contenido para redes sociales basada en el artículo sobre: {topic}

        **CONTENIDO A GENERAR:**

        1. **TWITTER/X** (3 tweets diferentes):
           - Tweet 1: Hook + dato impactante (max 280 caracteres)
           - Tweet 2: Perspectiva diferente + pregunta (max 280 caracteres)
           - Tweet 3: Call-to-action + enlace (max 280 caracteres)

        2. **FACEBOOK**:
           - Post engaging con pregunta para generar interacción
           - Incluye emoji relevante y call-to-action
           - 100-150 palabras

        3. **LINKEDIN**:
           - Post profesional con insights clave
           - Enfoque en implicaciones business/profesionales
           - 150-200 palabras

        4. **INSTAGRAM STORIES**:
           - Texto corto y visual para historia
           - Incluye pregunta interactiva
           - 30-40 palabras máximo

        5. **HASHTAGS ESTRATÉGICOS**:
           - 8-12 hashtags relevantes y trending
           - Mix de hashtags populares y nicho
           - Incluye hashtags locales (Colombia)

        **ESTRATEGIA ADICIONAL:**
        - Mejores horarios para publicar en cada plataforma
        - Tipo de imágenes/videos recomendados
        - Estrategia de engagement y respuesta
        - KPIs a monitorear
        """,
        agent=social_agent,
        expected_output="Estrategia completa de redes sociales con contenido para múltiples plataformas",
        context=[editing_task]
    )

    return [research_task, writing_task, editing_task, social_task]

# =============================================================================
# FUNCIÓN PRINCIPAL
# =============================================================================

def ejecutar_redaccion_digital(tema):
    """Ejecuta el flujo completo de la redacción digital"""

    print(f"🏢 INICIANDO REDACCIÓN DIGITAL CON OPENAI")
    print(f"📰 Tema: {tema}")
    print(f"🤖 Modelo: ChatGPT-4o-mini")
    print("="*60)

    # Inicializar tracker global
    global global_tracker
    global_tracker = NewsroomTracker()

    # Configurar agentes
    print("⚙️ Configurando agentes...")
    agents = setup_newsroom_agents()

    # Crear tareas
    print("📋 Creando tareas...")
    tasks = create_newsroom_tasks(agents, tema)

    # Configurar crew
    print("🎯 Configurando crew...")
    newsroom_crew = Crew(
        agents=list(agents),
        tasks=tasks,
        verbose=True
    )

    # Ejecutar flujo de trabajo
    print("🚀 Ejecutando flujo de trabajo...")
    print("   1️⃣ Investigación en curso...")
    print("   2️⃣ Redacción en espera...")
    print("   3️⃣ Edición en espera...")
    print("   4️⃣ Social media en espera...")
    print("-"*60)

    # Ejecutar y capturar resultados
    crew_result = newsroom_crew.kickoff()

    # Obtener resultados individuales de cada tarea
    task_results = {}
    if hasattr(crew_result, 'tasks_output') and crew_result.tasks_output:
        for i, task_output in enumerate(crew_result.tasks_output):
            task_names = ['investigacion', 'articulo', 'edicion', 'redes_sociales']
            if i < len(task_names):
                task_results[task_names[i]] = task_output.raw if hasattr(task_output, 'raw') else str(task_output)
    else:
        # Fallback: intentar acceder a las tareas directamente
        for i, task in enumerate(tasks):
            task_names = ['investigacion', 'articulo', 'edicion', 'redes_sociales']
            if i < len(task_names) and hasattr(task, 'output'):
                task_results[task_names[i]] = task.output.raw if hasattr(task.output, 'raw') else str(task.output)

    # Si no se pudieron obtener resultados individuales, usar el resultado final
    if not task_results:
        task_results['redes_sociales'] = str(crew_result)

    # Generar reporte final completo
    final_report = f"""
# 📰 PRODUCCIÓN EDITORIAL COMPLETA
## Tema: {tema}
## Modelo: ChatGPT-4o-mini (OpenAI)
{'='*80}

## 🔍 1. INVESTIGACIÓN REALIZADA
{task_results.get('investigacion', 'No disponible')}

{'='*60}

## ✍️ 2. ARTÍCULO REDACTADO
{task_results.get('articulo', 'No disponible')}

{'='*60}

## 📝 3. ARTÍCULO EDITADO
{task_results.get('edicion', 'No disponible')}

{'='*60}

## 📱 4. CONTENIDO PARA REDES SOCIALES
{task_results.get('redes_sociales', str(crew_result))}

{'='*80}
{global_tracker.get_sources_summary()}

---
✅ **PROCESO COMPLETADO EXITOSAMENTE**
🤖 4 agentes especializados trabajaron en colaboración
📊 {len(global_tracker.sources)} fuentes consultadas y verificadas
🔗 **Pipeline ejecutado**: Investigación → Redacción → Edición → Social Media
⚡ **Modelo utilizado**: ChatGPT-4o-mini para máxima estabilidad
📋 **Tareas completadas**: {len([k for k, v in task_results.items() if v != 'No disponible'])}/4
"""

    return final_report

# =============================================================================
# EJEMPLO DE USO
# =============================================================================

if __name__ == "__main__":
    # Ejecutar con tema específico
    resultado = ejecutar_redaccion_digital("Nuevas regulaciones de IA en Colombia 2025")
    print(resultado)